In [1]:
!nvidia-smi

Mon Oct  5 06:26:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers datasets accelerate sentencepiece gradio

In [3]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)

from datasets import Dataset
import torch

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [5]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model = model.to(device)

print("Model loaded successfully!")


config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded successfully!


In [6]:
data = {
    "input": [
        "What is artificial intelligence?",
        "What is machine learning?",
        "What is deep learning?",
        "What is generative AI?",
        "What is a neural network?",
        "What is NLP?",
        "What is a transformer model?",
        "What is a chatbot?",
        "What is computer vision?",
        "What is reinforcement learning?"
    ],

    "output": [
        "Artificial intelligence is the field of creating machines that can perform tasks requiring human intelligence.",
        "Machine learning is a branch of AI where machines learn patterns from data.",
        "Deep learning is a type of machine learning that uses neural networks with multiple layers.",
        "Generative AI is AI that can create new content such as text, images, audio, or code.",
        "A neural network is a computing model inspired by the human brain that learns patterns from data.",
        "NLP is a field of AI that enables computers to understand and process human language.",
        "A transformer is a neural network architecture that uses attention mechanisms to process sequential data.",
        "A chatbot is an AI application that communicates with users using natural language.",
        "Computer vision enables computers to understand and analyze images and videos.",
        "Reinforcement learning is a machine learning technique where an agent learns through rewards and penalties."
    ]
}

dataset = Dataset.from_dict(data)

print(dataset)

Dataset({
    features: ['input', 'output'],
    num_rows: 10
})


In [7]:
def preprocess_function(examples):

    inputs = tokenizer(
        examples["input"],
        max_length=128,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["output"],
        max_length=128,
        truncation=True
    )

    inputs["labels"] = labels["input_ids"]

    return inputs

In [8]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True
)

print(tokenized_dataset)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Dataset({
    features: ['input', 'output', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 10
})


In [9]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [10]:
training_args = TrainingArguments(
    output_dir="./flan_t5_finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none"
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Trainer created successfully!")

Trainer created successfully!


In [13]:
trainer.train()

Step,Training Loss
1,3.374943
2,3.078665
3,3.091547
4,3.130921
5,2.967664
6,2.944001
7,2.873351
8,2.842235
9,3.264959
10,2.541151


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15, training_loss=2.9351762453715007, metrics={'train_runtime': 22.2966, 'train_samples_per_second': 1.345, 'train_steps_per_second': 0.673, 'total_flos': 78422556672.0, 'train_loss': 2.9351762453715007, 'epoch': 3.0})

In [14]:
model.save_pretrained("./flan_t5_finetuned")
tokenizer.save_pretrained("./flan_t5_finetuned")

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [15]:
!ls ./flan_t5_finetuned

checkpoint-10  checkpoint-5  generation_config.json  tokenizer_config.json
checkpoint-15  config.json   model.safetensors	     tokenizer.json


In [16]:
def generate_text(question):

    inputs = tokenizer(
        question,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_length=100
        )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return response

In [17]:
print(generate_text("What is artificial intelligence?"))

Artificial intelligence is a technology that uses technology to measure the speed of the process of measuring the speed of an object.


In [18]:
print(generate_text("What is generative AI?"))

generative AI is a process that uses the knowledge of a machine to develop a model of generative reasoning.


In [19]:
print(generate_text("What is machine learning?"))

Machine learning is a process in machine learning.


In [20]:
import gradio as gr

In [21]:
def chatbot(question):

    return generate_text(question)

In [22]:
demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        label="Enter your question"
    ),
    outputs=gr.Textbox(
        label="Generated Answer"
    ),
    title="Fine-Tuned Generative AI Model",
    description="Ask a question to the fine-tuned FLAN-T5 model."
)

In [23]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://889fc9223d55b574ff.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
